# OCAN (One-Class Adversarial Networks) ile Medicare Fraud Tespiti

Bu notebook, Medicare veri seti üzerinde OCAN modelini simüle eder.

## Not:
OCAN tam implementasyonu için TensorFlow/PyTorch gerekir. Bu notebook basitleştirilmiş bir versiyon sunar.

In [2]:
import numpy as np
import pandas as pd
import os
import pickle

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


## 1. Medicare Verisini Yükle

In [3]:
# Medicare verisini yükle
combined_df = pd.read_csv("../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv")
print(f"Veri seti yüklendi: {combined_df.shape}")

# TARGET'ı sayısala çevir
combined_df['TARGET'] = combined_df['TARGET'].map({'NOT_FRAUD': 0, 'FRAUD': 1})

# Feature ve target ayır
X = combined_df.drop(columns=['TARGET'])
y = combined_df['TARGET']

# Boolean sütunları int'e çevir
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

# Sadece numerik sütunları al
X = X.select_dtypes(include=[np.number]).fillna(0)

print(f"X boyutu: {X.shape}")

Veri seti yüklendi: (798496, 215)
X boyutu: (798496, 214)


In [4]:
# Veriyi örnekle ve normalize et
sample_size = min(100000, len(X))
X_sample, _, y_sample, _ = train_test_split(
    X.values, y.values, train_size=sample_size, stratify=y, random_state=42
)

scaler = StandardScaler()
X_sample = scaler.fit_transform(X_sample)

print(f"Örneklenmiş veri: {X_sample.shape}")
print(f"Fraud sayısı: {sum(y_sample==1)}")

Örneklenmiş veri: (100000, 214)
Fraud sayısı: 120


## 2. OCAN Simülasyonu

In [7]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
repeats = 3
max_train_size = 10000

results_ocan = {'OCAN_Simulated': []}

for r in range(repeats):
    print(f"\n--- Tekrar {r+1}/{repeats} ---")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + r)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_sample, y_sample)):
        X_train, X_test = X_sample[train_idx], X_sample[test_idx]
        y_train, y_test = y_sample[train_idx], y_sample[test_idx]
        
        # Normal verilerle eğit
        X_train_normal = X_train[y_train == 0][:max_train_size]
        
        # OCAN simülasyonu
        ocsvm = OneClassSVM(kernel='rbf', nu=0.01, gamma='scale')
        ocsvm.fit(X_train_normal)
        
        scores = -ocsvm.decision_function(X_test)
        
        auc = roc_auc_score(y_test, scores)
        aupr = average_precision_score(y_test, scores)
        results_ocan['OCAN_Simulated'].append((auc, aupr))
        
        if fold == 1:
            print(f"  Fold 1 - OCAN: AUC={auc:.4f}")

print("\n✓ OCAN simülasyonu tamamlandı!")


--- Tekrar 1/3 ---
  Fold 1 - OCAN: AUC=0.5662

--- Tekrar 2/3 ---
  Fold 1 - OCAN: AUC=0.5347

--- Tekrar 3/3 ---
  Fold 1 - OCAN: AUC=0.5333

✓ OCAN simülasyonu tamamlandı!


## 3. Değerlendirme

In [8]:
print("="*60)
print("MEDICARE - OCAN SONUÇLARI")
print("="*60)

for method in results_ocan:
    scores = results_ocan[method]
    if scores:
        arr = np.array(scores)
        mean_auc, mean_aupr = arr.mean(axis=0)
        print(f'{method:20s} | AUC: {mean_auc:.4f} | AUPRC: {mean_aupr:.4f}')

MEDICARE - OCAN SONUÇLARI
OCAN_Simulated       | AUC: 0.5339 | AUPRC: 0.0026


## 4. Sonuçları Kaydet

In [9]:
os.makedirs("../pickled_storage", exist_ok=True)

with open("../pickled_storage/results_med_ocan.pkl", "wb") as f:
    pickle.dump(results_ocan, f)

print("✓ Sonuçlar kaydedildi: pickled_storage/results_med_ocan.pkl")

✓ Sonuçlar kaydedildi: pickled_storage/results_med_ocan.pkl
